# Diagnose SCL masking

Explore how alternative SCL masks affect valid observations in Nahal Rubin.
This optional notebook makes one imagery request per configured window and does
not modify the composite outputs. For the acquisition method, see the
[Imagery Acquisition and Preparation white paper](../docs/white-papers/Imagery-Acquisition-and-Preparation.md).


In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.validation import make_valid

from s2grab import S2Client, make_scl_count_evalscript, month_windows
from s2grab.config import AcquisitionConfig
from s2grab.sites import load_aois

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"

ACQUISITION = AcquisitionConfig()
YEARS = list(ACQUISITION.years)
MONTHS = list(ACQUISITION.months)
MAX_CLOUD = ACQUISITION.max_cloud
UPSAMPLING = ACQUISITION.upsampling
MASK_SCL = list(ACQUISITION.mask_scl)
SITES_SOURCE = ROOT / "sites" / "StudySites.gpkg"
ID_FIELD = "site_name"
OUT_DIR = ROOT / "outputs" / "composites"

sites = gpd.read_file(SITES_SOURCE)
invalid = ~sites.geometry.is_valid
if invalid.any():
    sites.loc[invalid, "geometry"] = sites.loc[invalid, "geometry"].apply(make_valid)

aois = load_aois(sites, id_field=ID_FIELD, resolution=ACQUISITION.resolution)
client = S2Client()


In [9]:
# --- Fetch per-pixel SCL class counts for NahalRubin (one ORBIT request/window)
# Returns a 13-band raster per window: ["valid", "SCL_0" ... "SCL_11"].

SITE = "NahalRubin"
site_aoi = aois[SITE]
scl_windows = month_windows(YEARS, MONTHS)

scl_product = make_scl_count_evalscript()
SCL_BANDS = list(scl_product.band_names)                       # 13 band names
band_idx = {int(n.split("_")[1]): i                            # SCL class -> band index
            for i, n in enumerate(SCL_BANDS) if n.startswith("SCL_")}

scl_counts = {}                                                # window label -> (H, W, 13)
for w in scl_windows:
    img = client.fetch(
        site_aoi, start=w["start"], end=w["end"],
        product=scl_product, max_cloud=MAX_CLOUD, upsampling=UPSAMPLING,
    )
    if img is None:
        print(f"{SITE} {w['label']}: no scenes")
        continue
    scl_counts[w["label"]] = img.array
    print(f"{SITE} {w['label']}: up to {int(np.nanmax(img.array[:, :, 0]))} valid obs/pixel")

print(f"\n{len(scl_counts)} window(s) fetched")

NahalRubin 2019-01: up to 4 valid obs/pixel
NahalRubin 2019-05: up to 5 valid obs/pixel
NahalRubin 2019-09: up to 4 valid obs/pixel
NahalRubin 2019-11: up to 5 valid obs/pixel
NahalRubin 2020-01: up to 3 valid obs/pixel
NahalRubin 2020-05: up to 4 valid obs/pixel
NahalRubin 2020-09: up to 6 valid obs/pixel
NahalRubin 2020-11: up to 2 valid obs/pixel
NahalRubin 2021-01: up to 1 valid obs/pixel
NahalRubin 2021-05: up to 6 valid obs/pixel
NahalRubin 2021-09: up to 4 valid obs/pixel
NahalRubin 2021-11: up to 3 valid obs/pixel
NahalRubin 2022-01: up to 3 valid obs/pixel
NahalRubin 2022-05: up to 5 valid obs/pixel
NahalRubin 2022-09: up to 6 valid obs/pixel
NahalRubin 2022-11: up to 2 valid obs/pixel
NahalRubin 2023-01: up to 2 valid obs/pixel
NahalRubin 2023-05: up to 5 valid obs/pixel
NahalRubin 2023-09: up to 6 valid obs/pixel
NahalRubin 2023-11: up to 4 valid obs/pixel
NahalRubin 2024-01: up to 1 valid obs/pixel
NahalRubin 2024-05: up to 4 valid obs/pixel
NahalRubin 2024-09: up to 3 vali

In [10]:
# --- Simulate candidate SCL mask sets offline (no refetching) ----------------
# Each candidate lists the classes to DROP. A pixel is no-data under a set when
# it has zero clear observations left: valid - sum(counts of masked classes) == 0.
mask_candidates = {
    "current":     set(MASK_SCL),                 # [0,1,2,3,7,8,9,10,11]
    "lib_default": {0, 1, 3, 8, 9, 10, 11},       # keeps 2 (dark) and 7 (unclassified)
    "keep_2":      set(MASK_SCL) - {2},           # stop dropping dark/topographic shadow
    "keep_2_7":    set(MASK_SCL) - {2, 7},
    "keep_2_3_7":  set(MASK_SCL) - {2, 3, 7},     # also keep cloud shadow
    "clouds_only": {0, 1, 8, 9, 10},              # only bad pixels + true clouds
}


def simulate_nodata(stack, mask_set, study):
    """% of study-area pixels with zero clear observations under mask_set."""
    valid = stack[:, :, 0]
    masked = sum((stack[:, :, band_idx[c]] for c in mask_set), np.zeros_like(valid))
    clear = valid - masked
    n = int(study.sum())
    return 100.0 * int(((clear <= 0) & study).sum()) / n if n else float("nan")


rows = []
for label, stack in scl_counts.items():
    study = site_aoi.footprint
    rows.append({"window": label,
                 **{cand: simulate_nodata(stack, mset, study)
                    for cand, mset in mask_candidates.items()}})

sim = pd.DataFrame(rows).set_index("window").sort_index()
print("mean % no-data across windows, by candidate mask set:")
print(sim.mean().round(3).sort_values().to_string())
sim.round(3)

mean % no-data across windows, by candidate mask set:
lib_default    0.000
keep_2_7       0.000
clouds_only    0.000
keep_2_3_7     0.000
keep_2         0.304
current        0.304


,current,lib_default,keep_2,keep_2_7,keep_2_3_7,clouds_only
window,,,,,,
2019-01,0.000,0.0,0.000,0.0,0.0,0.0
2019-05,0.000,0.0,0.000,0.0,0.0,0.0
2019-09,0.852,0.0,0.852,0.0,0.0,0.0
2019-11,0.000,0.0,0.000,0.0,0.0,0.0
2020-01,0.000,0.0,0.000,0.0,0.0,0.0
2020-05,0.000,0.0,0.000,0.0,0.0,0.0
2020-09,0.254,0.0,0.254,0.0,0.0,0.0
2020-11,0.000,0.0,0.000,0.0,0.0,0.0
2021-01,0.000,0.0,0.000,0.0,0.0,0.0


In [11]:
# --- Validate: "current" simulation should reproduce the pipeline's pct_nodata
# (small diffs are expected from SCL 20 m -> 10 m resampling onto the output grid).
qa = pd.read_csv(OUT_DIR / "qa_manifest.csv")
qa = qa[qa["site"] == SITE].set_index("window")["pct_nodata"]
check = pd.DataFrame({"manifest": qa, "simulated_current": sim["current"]})
check["abs_diff"] = (check["manifest"] - check["simulated_current"]).abs()
check.round(3)

,manifest,simulated_current,abs_diff
window,,,
2019-01,0.000,0.000,0.0
2019-05,0.000,0.000,0.0
2019-09,0.852,0.852,0.0
2019-11,0.000,0.000,0.0
2020-01,0.000,0.000,0.0
2020-05,0.000,0.000,0.0
2020-09,0.254,0.254,0.0
2020-11,0.000,0.000,0.0
2021-01,0.000,0.000,0.0


In [12]:
# --- Observation-weighted SCL class mix per window --------------------------
# What % of ALL valid observations over the study area fell in each class.
# Shows which classes are prevalent (and thus candidates to stop masking).
mix_rows = []
for label, stack in scl_counts.items():
    study = site_aoi.footprint
    total = float(stack[:, :, 0][study].sum())
    mix_rows.append({"window": label,
                     **{f"SCL_{c}": (100.0 * float(stack[:, :, i][study].sum()) / total
                                     if total else np.nan)
                        for c, i in band_idx.items()}})

mix = pd.DataFrame(mix_rows).set_index("window").sort_index()
mix = mix.loc[:, (mix.fillna(0) > 0).any()]   # drop classes that never appear
mix.round(3)

,SCL_2,SCL_3,SCL_4,SCL_5,SCL_6,SCL_7,SCL_8,SCL_9,SCL_10
window,,,,,,,,,
2019-01,0.000,0.000,45.579,54.408,0.013,0.000,0.000,0.000,0.000
2019-05,0.000,0.000,19.069,60.285,0.000,0.646,3.496,16.504,0.000
2019-09,0.000,0.000,18.556,79.205,0.000,2.239,0.000,0.000,0.000
2019-11,0.000,0.000,26.117,73.873,0.010,0.000,0.000,0.000,0.000
2020-01,0.000,0.000,44.805,55.161,0.034,0.000,0.000,0.000,0.000
2020-05,0.000,2.052,24.730,64.466,0.000,1.196,3.912,3.645,0.000
2020-09,0.000,2.752,11.751,82.981,0.000,2.515,0.000,0.000,0.000
2020-11,0.000,32.850,22.767,44.383,0.000,0.000,0.000,0.000,0.000
2021-01,0.000,0.000,42.087,57.913,0.000,0.000,0.000,0.000,0.000
